In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sample = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

print(train.shape)
print(test.shape)

train.head()

In [ ]:
train["text"] = (
    "Question: " + train["prompt"].fillna("") +
    " A: " + train["A"].fillna("") +
    " B: " + train["B"].fillna("") +
    " C: " + train["C"].fillna("") +
    " D: " + train["D"].fillna("") +
    " E: " + train["E"].fillna("")
)

test["text"] = (
    "Question: " + test["prompt"].fillna("") +
    " A: " + test["A"].fillna("") +
    " B: " + test["B"].fillna("") +
    " C: " + test["C"].fillna("") +
    " D: " + test["D"].fillna("") +
    " E: " + test["E"].fillna("")
)

In [ ]:
label2id = {
    "A":0,
    "B":1,
    "C":2,
    "D":3,
    "E":4
}

id2label = {v:k for k,v in label2id.items()}

train["label"] = train["answer"].map(label2id)

In [ ]:
train.head()

In [ ]:
print(train["answer"].value_counts(normalize=True))

In [ ]:
print(train.iloc[0]["prompt"])
print(train.iloc[0]["A"])
print(train.iloc[0]["B"])
print(train.iloc[0]["C"])
print(train.iloc[0]["D"])
print(train.iloc[0]["E"])
print(train.iloc[0]["answer"])

In [ ]:
from transformers import AutoTokenizer

MODEL = "microsoft/deberta-v3-large"


tokenizer = AutoTokenizer.from_pretrained(MODEL)



In [ ]:
from datasets import Dataset

train_ds = Dataset.from_pandas(
    train[["text","label"]]
)

test_ds = Dataset.from_pandas(
    test[["text"]]
)

In [ ]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

In [ ]:
from transformers import (
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=5
)

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="no",
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds
)

trainer.train()

In [ ]:
import numpy as np
import torch

pred = trainer.predict(test_ds)

probs = torch.softmax(
    torch.tensor(pred.predictions),
    dim=1
).numpy()

predictions = []

for p in probs:
    top3 = np.argsort(p)[::-1][:3]
    predictions.append(
        " ".join([id2label[i] for i in top3])
    )

In [ ]:
submission = pd.DataFrame({
    "id": test["id"],
    "Prediction": predictions
})

submission.to_csv(
    "submission.csv",
    index=False
)

submission.head()